# S0_multi: concise tutorial

This notebook demonstrates the main workflow with synthetic ternary data:

1. convert $S^0$ to chemical-potential gradients;
2. detect inconsistent trajectory splits;
3. prepare gradient observations for a Gaussian process.

The assertions also act as a small regression test.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from szero import (
    find_s0_split_outlier_rows,
    plot_s0_split_residuals,
    prepare_gp_gradient_data,
    s0_to_gamma,
)

np.set_printoptions(precision=6, suppress=True)

## 1. From S0 to Gamma

For components `[A, B, C]`, choose two independent coordinates, here `[x_A, x_B]`; normalization determines $x_C$. The output follows

$$\Gamma_{\alpha j}=\partial\mu_\alpha/\partial\widetilde{x}_j,$$

so rows are chemical-potential species and columns are composition directions. In the ideal limit (diagonal S0 equal to one), the excess gradient should vanish.

In [ ]:
components = ["A", "B", "C"]
independent = ["A", "B"]
temperature = 303.15

ideal = {
    "x_A": 0.2, "x_B": 0.3, "x_C": 0.5,
    "S_AA": 1.0, "S_AB": 0.0, "S_AC": 0.0,
    "S_BB": 1.0, "S_BC": 0.0, "S_CC": 1.0,
}
ideal_gamma = s0_to_gamma(
    ideal, independent, components=components,
    temperature=temperature, excess=True, return_labels=False,
)
assert np.allclose(ideal_gamma, 0.0, atol=1e-10)

entry = ideal | {
    "S_AA": 1.2, "S_AB": 0.10, "S_AC": 0.05,
    "S_BB": 1.4, "S_BC": 0.12, "S_CC": 1.1,
}
gamma, row_labels, column_labels = s0_to_gamma(
    entry, independent, components=components,
    temperature=temperature, excess=True,
)
print("rows:", row_labels, "columns:", column_labels)
print(gamma)

x = np.array([entry[f"x_{name}"] for name in components])
assert gamma.shape == (3, 2)
assert np.allclose(x @ gamma, 0.0, atol=1e-10)  # Gibbs-Duhem

## 2. Check trajectory splits

Split columns use names such as `split0_AB_S0`. The synthetic table below contains one deliberate outlier in `row_id=2`. Residual plots help reveal disagreement before split values are averaged.

In [ ]:
pairs = ["AA", "AB", "AC", "BB", "BC", "CC"]
base_s0 = dict(zip(pairs, [1.2, 0.10, 0.05, 1.4, 0.12, 1.1]))
compositions = [(0.1, 0.2, 0.7), (0.2, 0.3, 0.5), (0.3, 0.4, 0.3), (0.5, 0.3, 0.2)]

rows = []
for row_id, (x_a, x_b, x_c) in enumerate(compositions):
    row = {"row_id": row_id, "x_A": x_a, "x_B": x_b, "x_C": x_c}
    for split, offset in enumerate([-0.01, 0.0, 0.01]):
        for pair in pairs:
            row[f"split{split}_{pair}_S0"] = base_s0[pair] + offset
    rows.append(row)

df = pd.DataFrame(rows)
df.loc[df.row_id == 2, "split2_AB_S0"] += 0.8

fig, _ = plot_s0_split_residuals(df, x_col="row_id")
plt.show()

bad_mask, details = find_s0_split_outlier_rows(
    df, max_diff=0.3, percent_min_diff=0.1, percent_threshold=0.75
)
clean_df = df.loc[~bad_mask].copy()
print(pd.DataFrame(details)[["row_id", "pair", "worst_column", "reason"]])
assert df.loc[bad_mask, "row_id"].tolist() == [2]

## 3. Prepare GP gradient observations

`column_map` maps canonical keys to table columns. Mapping an S0 key to a list averages the clean split estimates. For species A, `G_g[i]` contains `[d mu_A/dx_A, d mu_A/dx_B]`.

In [ ]:
column_map = {"x_A": "x_A", "x_B": "x_B", "x_C": "x_C"}
for pair in pairs:
    column_map[f"S_{pair}"] = [f"split{s}_{pair}_S0" for s in range(3)]

X_g, G_g, grad_indices = prepare_gp_gradient_data(
    clean_df,
    components=components,
    independent_components=independent,
    column_map=column_map,
    species="A",
    temperature=temperature,
    excess=True,
)

print("X_g:\n", X_g)
print("G_g:\n", G_g)
print("grad_indices:\n", grad_indices)

assert X_g.shape == G_g.shape == (len(clean_df), 2)
assert grad_indices.shape == (len(clean_df) * 2, 2)

## Takeaways

The main workflow is `split CSV -> diagnose/clean -> average splits -> Gamma -> GP gradients`. Pass component order explicitly, use interior compositions with `excess=True`, and choose cleaning thresholds based on the uncertainty scale of the data.